# Solar Energy Analytics & Predictive Maintenance

This notebook analyzes solar PV generation data, explores relationships between environmental variables and AC power output, trains Random Forest regression models for power prediction, and applies a rule-based performance/cleaning detection workflow.

## Workflow

1. Load and merge generation and weather data.
2. Perform exploratory analysis and visualization.
3. Engineer time-based features.
4. Train and evaluate Random Forest regression models.
5. Generate predicted power and analyze power loss.
6. Identify potential cleaning/maintenance cases using the implemented rule-based logic.
7. Analyze feature importance and descriptive statistics.

> **Reproducibility:** Run this notebook from the repository root or configure the project path in the setup cell below.


In [ ]:
from pathlib import Path
import sys

# Repository paths
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = Path("..").resolve()

DATA_DIR = PROJECT_ROOT / "data"
RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")


In [ ]:
#Solar Power Generation Forecasting and Analysis using Machine Learning
#Dataset: Solar Power Generation Dataset (Kaggle)
#Objective: Predict solar PV power output using weather and irradiation data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print("Environment ready!")

In [ ]:

from src.data_loader import load_and_merge_data

df = load_and_merge_data(
    DATA_DIR / "Plant_1_Generation_Data.csv",
    DATA_DIR / "Plant_1_Weather_Sensor_Data.csv"
)

df.head()

In [ ]:
df["DATE_TIME"] = pd.to_datetime(df["DATE_TIME"])

In [ ]:
import os

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

## Exploratory Data Analysis

In [ ]:
#Plot 1: Solar Power Generation Over Time
import matplotlib.pyplot as plt

sample = df[df["SOURCE_KEY_x"] == df["SOURCE_KEY_x"].iloc[0]]

plt.figure(figsize=(12,5))
plt.plot(sample["DATE_TIME"], sample["AC_POWER"])

plt.title("Solar Power Generation Over Time (Single Inverter)")
plt.xlabel("Time")
plt.ylabel("AC Power")
plt.savefig(RESULTS_DIR / "power_generation_over_time.png")
plt.show()

In [ ]:
#Plot 2: Irradiance vs Power Output
import seaborn as sns

plt.figure(figsize=(8,5))

sns.scatterplot(x=df["IRRADIATION"], y=df["AC_POWER"])

plt.title("Solar Irradiation vs Power Output")
plt.xlabel("Solar Irradiation")
plt.ylabel("AC Power")
plt.savefig(RESULTS_DIR / "irradiation_vs_power.png")
plt.show()

In [ ]:
#Plot 3: Temperature vs Power
plt.figure(figsize=(8,5))

sns.scatterplot(x=df["MODULE_TEMPERATURE"], y=df["AC_POWER"])

plt.title("Module Temperature vs Power Output")
plt.xlabel("Module Temperature")
plt.ylabel("AC Power")
plt.savefig(RESULTS_DIR / "module_temperature_vs_power.png")
plt.show()

In [ ]:
df["hour"] = df["DATE_TIME"].dt.hour

In [ ]:
#Plot 4: Hourly Solar Generation
hourly = df.groupby("hour")["AC_POWER"].mean()

plt.figure(figsize=(10,5))
hourly.plot(kind="bar")

plt.title("Average Solar Power Generation by Hour")
plt.xlabel("Hour of Day")
plt.ylabel("Average AC Power")
plt.savefig(RESULTS_DIR / "hourly_average_power.png")
plt.show()

In [ ]:
features = df[[
    "AC_POWER",
    "IRRADIATION",
    "AMBIENT_TEMPERATURE",
    "MODULE_TEMPERATURE",
    "hour"
]]

features.head()

In [ ]:
features.to_csv(DATA_DIR / "solar_cleaned_data.csv", index=False)

## Solar Power Prediction

In [ ]:
#Model training part
features = df[[
    "IRRADIATION",
    "AMBIENT_TEMPERATURE",
    "MODULE_TEMPERATURE",
    "hour"
]]

target = df["AC_POWER"]

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    features,
    target,
    test_size=0.2,
    random_state=42
)

In [ ]:
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)

In [ ]:
import joblib
import os

# create results folder if it doesn't exist
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# save trained model
joblib.dump(model, RESULTS_DIR / "solar_power_model.pkl")

print("Model saved successfully!")

In [ ]:
predictions = model.predict(X_test)

In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score

mae = mean_absolute_error(y_test, predictions)
r2 = r2_score(y_test, predictions)

print("MAE:", mae)
print("R2 Score:", r2)

In [ ]:
plt.figure(figsize=(8,6))

plt.scatter(y_test, predictions)

plt.xlabel("Actual Power")
plt.ylabel("Predicted Power")
plt.title("Actual vs Predicted Solar Power")
plt.savefig(RESULTS_DIR / "actual_vs_predicted_power.png")
plt.show()

## Time-Based Features and Prediction

In [ ]:
#creating time based features
df["day"] = df["DATE_TIME"].dt.day
df["month"] = df["DATE_TIME"].dt.month
df["minute"] = df["DATE_TIME"].dt.minute

In [ ]:
features = df[[
    "IRRADIATION",
    "AMBIENT_TEMPERATURE",
    "MODULE_TEMPERATURE",
    "hour",
    "day",
    "month"
]]

target = df["AC_POWER"]

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    features,
    target,
    test_size=0.2,
    random_state=42
)

model = RandomForestRegressor(n_estimators=150, random_state=42)
model.fit(X_train, y_train)

In [ ]:
import pandas as pd

future_data = pd.DataFrame({
    "IRRADIATION":[0.9],
    "AMBIENT_TEMPERATURE":[30],
    "MODULE_TEMPERATURE":[45],
    "hour":[12],
    "day":[15],
    "month":[6]
})

prediction = model.predict(future_data)

print("Predicted Solar Power:", prediction[0])

## Performance Monitoring and Cleaning Detection

In [ ]:
#Solar Panel Performance Monitoring / Cleaning Detection
df["PREDICTED_POWER"] = model.predict(features)
df["POWER_LOSS"] = df["PREDICTED_POWER"] - df["AC_POWER"]

In [ ]:
maintenance_cases = df[
    (df["IRRADIATION"] > 0.7) & 
    (df["POWER_LOSS"] > 200)
]

maintenance_cases.head()

In [ ]:
plt.figure(figsize=(8,5))

plt.scatter(df["IRRADIATION"], df["POWER_LOSS"])

plt.xlabel("Solar Irradiation")
plt.ylabel("Power Loss")
plt.title("Solar Panel Performance Loss Detection")

plt.show()

In [ ]:
df["EFFICIENCY_RATIO"] = df["AC_POWER"] / df["PREDICTED_POWER"]

In [ ]:
from src.cleaning_detection import detect_cleaning

df = detect_cleaning(df)

In [ ]:
df[df["CLEANING_REQUIRED"] == True].head()

## Feature Importance and Results Export

In [ ]:
import pandas as pd

importance = pd.Series(model.feature_importances_, index=features.columns)

importance.sort_values().plot(kind='barh', figsize=(8,5))

plt.title("Feature Importance for Solar Power Prediction")
plt.xlabel("Importance")
plt.savefig(RESULTS_DIR / "feature_importance.png")
plt.show()

In [ ]:
df.to_csv(RESULTS_DIR / "solar_prediction_results.csv", index=False)

## Statistical and Correlation Analysis

In [ ]:
#statistical analysis
df.describe()
df["AC_POWER"].describe()
df["AC_POWER"].mode()

#Correlation Analysis
import seaborn as sns
import matplotlib.pyplot as plt

corr = df.select_dtypes(include=['number']).corr()

plt.figure(figsize=(8,6))
sns.heatmap(corr, annot=True, cmap="coolwarm")
plt.title("Correlation Matrix")
plt.show()

#Distribution Analysis
sns.histplot(df["AC_POWER"], kde=True)
plt.title("Distribution of AC Power")
plt.show()
#Feature vs Target Analysis
sns.scatterplot(x=df["IRRADIATION"], y=df["AC_POWER"])
plt.title("Irradiation vs AC Power")
plt.show()

#Temperature Impact
sns.scatterplot(x=df["MODULE_TEMPERATURE"], y=df["AC_POWER"])
plt.title("Module Temperature vs Power")
plt.show()




data = df["AC_POWER"]

mean = data.mean()
median = data.median()

plt.figure(figsize=(8,5))
sns.histplot(data, kde=True)

plt.axvline(mean, color='red', linestyle='--', label=f'Mean: {mean:.2f}')
plt.axvline(median, color='green', linestyle='--', label=f'Median: {median:.2f}')

plt.legend()
plt.title("AC Power Distribution with Mean & Median")
plt.show()

# Basic Statistics

print("Mean:", df["AC_POWER"].mean())
print("Median:", df["AC_POWER"].median())
print("Mode:", df["AC_POWER"].mode()[0])
print("Standard Deviation:", df["AC_POWER"].std())